In [0]:
# Databricks notebook source
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

securable_type = "CATALOG"
securable_full_name = "t3_a00cd1_snd_arg02_dbws01"
policy_name = "4TIMERESTRICTED"

policy_payload = {
    "on_securable_type": securable_type,
    "on_securable_fullname": securable_full_name,
    "name": policy_name,
    "comment": "",
    "to_principals": ["account users"],
    "for_securable_type": "TABLE",
    "policy_type": "POLICY_TYPE_ROW_FILTER",
    "except_principals": ["l205495@sandpit0x8a.onmicrosoft.com"],
    "when_condition": "hasTagValue(\"SHARINGMODE\",\"TIMERESTRICTED\")",
    "row_filter": {
        "function_name": f"{securable_full_name}.ext_parquet.business_hours_filter"
    }
}

# Check if policy exists
endpoint = f"/api/2.1/unity-catalog/policies/{securable_type}/{securable_full_name}/{policy_name}"
try:
    _ = w.api_client.do("GET", endpoint)
    # If GET succeeds, update the policy
    print(f"Updating existing policy: {policy_name}")
    w.api_client.do("PATCH", endpoint, data=json.dumps(policy_payload))
except Exception as e:
    if "NotFound" in str(e) or "RESOURCE_DOES_NOT_EXIST" in str(e):
        # If GET fails with not found, create the policy
        print(f"Creating new policy: {policy_name}")
        w.api_client.do("POST", "/api/2.1/unity-catalog/policies/", data=json.dumps(policy_payload))
    else:
        raise  # rethrow any other error
